In [67]:
import sys
from pathlib import Path
import aiohttp
import asyncio
import logging
from datetime import datetime, timedelta
import requests
from typing import List, Dict

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [68]:
from src_oop.core.database import Database
from src_oop.core.scraper import HTTPClient
import pandas as pd
from time import time, sleep

logger = logging.getLogger(__name__)

In [60]:
# def get_historical_stocks(
#     date_from: str = None,
#     date_to: str = None,
#     warehouse_id: int = 1,
#     page_size: int = 5000,
# ) -> List[Dict]:
#     """Получение исторических остатков с пагинацией"""
    
   
#     if date_from is None:
#         date_from = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
#     if date_to is None:
#         date_to = datetime.now().strftime("%Y-%m-%d")

 
#     url = "https://api-routing.star-vector.ru/api/warehouse_and_balances/get_historical_stocks"
    
#     all_res = []
#     page_num = 1
    
#     while True:
       
#         payload = {
#             "date_from": date_from,
#             "date_to": date_to,
#             "warehouse_id": warehouse_id,
#             "page_size": page_size,
#             "page_num": page_num
#         }
        
#         try:
            
#             res = requests.post(url, json=payload, timeout=30)
#             logger.debug(f"Отправлен запрос по получению сведений со страницы {page_num}")
            
#             if res.status_code == 200:
#                 data = res.json()
                
#                 # Если API возвращает список напрямую
#                 if isinstance(data, list):
#                     all_res.extend(data)
#                     print(f"✅ Страница {page_num}: получено {len(data)} записей")
                    
#                     # 🔴 УСЛОВИЕ ВЫХОДА: 
#                     if len(data) == 0:
#                         print(f"Получены все данные")
#                         break
#                 else:
#                     print(f"⚠️ Неожиданный формат ответа: {type(data)}")
#                     break
                    
#             elif res.status_code == 429:
#                 print("⚠️ Лимит запросов (429). Ожидание 10 секунд...")
#                 time.sleep(10)
#                 continue  # ⚠️ Повторяем тот же запрос, не увеличивая page_num
            
#             elif res.status_code >= 400:
#                 logger.error(f"❌ Ошибка API {res.status_code}: {res.text}")
#                 break
                
#         except requests.exceptions.RequestException as e:
#             print(f"❌ Ошибка соединения: {e}")
#             time.sleep(5)
#             continue
        
        
#         page_num += 1
        
        
#         if page_num > 100:
#             logger.warning("⚠️ Достигнут лимит страниц (100)")
#             break
    
#     logger.info(f"🏁 Всего собрано {len(all_res)} записей")
#     return all_res

In [ ]:
async def get_historical_stocks(
    session: aiohttp.ClientSession,
    date_from: str = None,
    date_to: str = None,
    warehouse_id: int = 1,
    page_size: int = 5000,
    api_key: Optional[str] = None,  # Если API требует ключ
) -> List[Dict]:
    """Получение исторических остатков с пагинацией"""
    
    # 1. Обработка дат
    if date_from is None:
        date_from = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
    if date_to is None:
        date_to = datetime.now().strftime("%Y-%m-%d")

    # 2. URL без пробелов
    url = "https://api-routing.star-vector.ru/api/warehouse_and_balances/get_historical_stocks"
    
    # 3. Создаем клиент (без токена, если API публичное)
    client = HTTPClient(
        session=session,
        api_key=api_key,
        account="StarVector",
        timeout=30.0  # Увеличил таймаут для тяжелых запросов
    )
    
    all_res = []
    page_num = 1
    
    while True:
        payload = {
            "date_from": date_from,
            "date_to": date_to,
            "warehouse_id": warehouse_id,
            "page_size": page_size,
            "page_num": page_num
        }
        
        # 4. Асинхронный POST-запрос через твой класс
        data = await client.post(url, json=payload, delay=1.0, retries=3)
        
        logger.debug(f"📄 Страница {page_num}: отправлен запрос")
        
        if data is None:
            logger.error(f"❌ Страница {page_num}: не получены данные")
            break
        
        # 5. Обработка ответа
        if isinstance(data, list):
            all_res.extend(data)
            logger.info(f"✅ Страница {page_num}: получено {len(data)} записей")
            
            # 🔴 УСЛОВИЕ ВЫХОДА
            if len(data) == 0:
                break
        else:
            logger.warning(f"⚠️ Неожиданный формат ответа: {type(data)}")
            break
        
        # 6. Следующая страница
        page_num += 1
        
        # 7. Защита от бесконечного цикла
        if page_num > 100:
            logger.warning("⚠️ Достигнут лимит страниц (100)")
            break
    
    logger.info(f"🏁 Всего собрано {len(all_res)} записей")
    return all_res

In [71]:
async def fetch_historical_stocks(date_from="2026-03-16", date_to="2026-03-16"):
    async with aiohttp.ClientSession() as session:
        data = await(get_historical_stocks(session, date_from=date_from, date_to=date_to))
        return data
    
data = await(fetch_historical_stocks())

In [72]:
data

[{'product_id': 'metawild_test',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 0}]},
 {'product_id': 'testwild',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 220}]},
 {'product_id': 'testwild2',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 110}]},
 {'product_id': 'testwild3',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 0}]},
 {'product_id': 'testwild4',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 0}]},
 {'product_id': 'wild0001517',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 0}]},
 {'product_id': 'wild0001535',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 0}]},
 {'product_id': 'wild100',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 0}]},
 {'product_id': 'wild1000',
  'data': [{'transaction_date': '2026-03-16', 'end_of_day_balance': 0}]},
 {'product_id': 'wild1001',
  'data': [{'transaction_date': '2026

In [ ]:
# data = get_historical_stocks(date_from="2025-10-20", date_to="2026-03-15")

✅ Страница 1: получено 1548 записей
✅ Страница 2: получено 1548 записей
✅ Страница 3: получено 1548 записей
✅ Страница 4: получено 1548 записей
✅ Страница 5: получено 1548 записей
✅ Страница 6: получено 1548 записей
✅ Страница 7: получено 1548 записей
✅ Страница 8: получено 1548 записей
✅ Страница 9: получено 1548 записей
✅ Страница 10: получено 1548 записей
✅ Страница 11: получено 1548 записей
✅ Страница 12: получено 1548 записей
✅ Страница 13: получено 1548 записей
✅ Страница 14: получено 1548 записей
✅ Страница 15: получено 1548 записей
✅ Страница 16: получено 1548 записей
✅ Страница 17: получено 1548 записей
✅ Страница 18: получено 1548 записей
✅ Страница 19: получено 1548 записей
✅ Страница 20: получено 1548 записей
✅ Страница 21: получено 1548 записей
✅ Страница 22: получено 1548 записей
✅ Страница 23: получено 1548 записей
✅ Страница 24: получено 1548 записей
✅ Страница 25: получено 1548 записей
✅ Страница 26: получено 1548 записей
✅ Страница 27: получено 1548 записей
✅ Страница

In [73]:
def process_historical_stocks(data: List)->List:
    if data:
        all_data = []
        for d in data:
            product_id = d['product_id']
            for i in d['data']:
                i['wild'] = product_id
                all_data.append(i)

        return pd.DataFrame(all_data)
    else:
        return pd.DataFrame()

In [74]:
df = process_historical_stocks(data)
df.tail(10)

,transaction_date,end_of_day_balance,wild
1538,2026-03-16,0,wild988
1539,2026-03-16,0,wild991
1540,2026-03-16,0,wild992
1541,2026-03-16,0,wild993
1542,2026-03-16,0,wild994
1543,2026-03-16,0,wild995
1544,2026-03-16,0,wild996
1545,2026-03-16,0,wild997
1546,2026-03-16,0,wild998
1547,2026-03-16,0,wild999


In [ ]:
engine = Database.get_engine()
table_name = "historical_stocks_fbs_service"
df.to_sql(table_name, engine)

556

In [ ]:
engine = Database.get_engine()

query = """ 
    WITH exp AS (
    SELECT
        end_date,
        -- Individual expense columns
        SUM(CASE WHEN type='Расходы офис' THEN value ELSE 0 END) AS "Расходы офис",
        SUM(CASE WHEN type='Услуги подбора персонала' THEN value ELSE 0 END) AS "Услуги подбора персонала",
        SUM(CASE WHEN type='Арендные платежи (офис)' THEN value ELSE 0 END) AS "Арендные платежи (офис)",
        SUM(CASE WHEN type='Услуги связи (интернет, телефон)' THEN value ELSE 0 END) AS "Услуги связи (интернет, телефон)",
        SUM(CASE WHEN type='Стоянка' THEN value ELSE 0 END) AS "Стоянка",
        SUM(CASE WHEN type='Эксплуатация здания' THEN value ELSE 0 END) AS "Эксплуатация здания",
        SUM(CASE WHEN type='Коммунальные платежи' THEN value ELSE 0 END) AS "Коммунальные платежи",
        SUM(CASE WHEN type='Расчету с поставщиком материалы' THEN value ELSE 0 END) AS "Расчету с поставщиком материалы",
        SUM(CASE WHEN type='Карго' THEN value ELSE 0 END) AS "Карго",
        SUM(CASE WHEN type='Консультационные услуги по оформлении сделки' THEN value ELSE 0 END) AS "Консультационные услуги по оформлении сделки",
        SUM(CASE WHEN type='Расходы склад' THEN value ELSE 0 END) AS "Расходы склад",
        SUM(CASE WHEN type='Транспортные расходы, гсм' THEN value ELSE 0 END) AS "Транспортные расходы, ГСМ",
        SUM(CASE WHEN type='Парковка' THEN value ELSE 0 END) AS "Парковка",
        SUM(CASE WHEN type='Обслуживание а/м' THEN value ELSE 0 END) AS "Обслуживание а/м",
        SUM(CASE WHEN type='Арендные платежи (склад)' THEN value ELSE 0 END) AS "Арендные платежи (склад)",
        SUM(CASE WHEN type='Страховка' THEN value ELSE 0 END) AS "Страховка",
        SUM(CASE WHEN type='ПО, сервисы, обслуживание ПО' THEN value ELSE 0 END) AS "ПО, сервисы, обслуживание ПО",
        SUM(CASE WHEN type='Реклама/продвижение на площадках МП' THEN value ELSE 0 END) AS "Реклама/продвижение на площадках МП",
        SUM(CASE WHEN type='Вб смена номера, перенос карточек' THEN value ELSE 0 END) AS "Вб смена номера, перенос карточек",
        SUM(CASE WHEN type='Услуги банка' THEN value ELSE 0 END) AS "Услуги банка",
        SUM(CASE WHEN type='Налог: НДФЛ' THEN value ELSE 0 END) AS "Налог: НДФЛ",
        SUM(CASE WHEN type='Налог: УСН' THEN value ELSE 0 END) AS "Налог: УСН",
        SUM(CASE WHEN type='Налог: СВ' THEN value ELSE 0 END) AS "Налог: СВ",
        SUM(CASE WHEN type='Налог: прочее' THEN value ELSE 0 END) AS "Налог: прочее",
        SUM(CASE WHEN type='Налог:НДС' THEN value ELSE 0 END) AS "Налог: НДС",
        SUM(CASE WHEN type='Штрафы, взыскания' THEN value ELSE 0 END) AS "Штрафы, взыскания",
        SUM(CASE WHEN type='Проценты кредит ВБ' THEN value ELSE 0 END) AS "Проценты кредит ВБ",
        SUM(CASE WHEN type='Проценты СИМПЛФИНАНС ООО МКК' THEN value ELSE 0 END) AS "Проценты СИМПЛФИНАНС",
        SUM(CASE WHEN type='Займ ВБ Проценты' THEN value ELSE 0 END) AS "Займ ВБ: проценты",
        SUM(CASE WHEN type='Кредит сберпроценты' THEN value ELSE 0 END) AS "Кредит Сбер: проценты",
        SUM(CASE WHEN type='Рови факторинг: проценты' THEN value ELSE 0 END) AS "Рови факторинг: проценты",
        SUM(CASE WHEN type='Рови факторинг:пени' THEN value ELSE 0 END) AS "Рови факторинг: пени",
        SUM(CASE WHEN type='Лизинг, покупка оборудования, помещений' THEN value ELSE 0 END) AS "Лизинг, покупка оборудования, помещений",
        SUM(CASE WHEN type='Заработная плата' THEN value ELSE 0 END) AS "Заработная плата",
        -- Sum of all above as 'Расходы компании'
        SUM(
            CASE WHEN type IN (
                'Расходы офис', 'Услуги подбора персонала', 'Арендные платежи (офис)',
                'Услуги связи (интернет, телефон)', 'Стоянка', 'Эксплуатация здания',
                'Коммунальные платежи', 'Расчету с поставщиком материалы', 'Карго',
                'Консультационные услуги по оформлении сделки', 'Расходы склад',
                'Транспортные расходы, гсм', 'Парковка', 'Обслуживание а/м',
                'Арендные платежи (склад)', 'Страховка', 'ПО, сервисы, обслуживание ПО',
                'Реклама/продвижение на площадках МП', 'Вб смена номера, перенос карточек',
                'Услуги банка', 'Налог: НДФЛ', 'Налог: УСН', 'Налог: СВ', 'Налог: прочее',
                'Налог:НДС', 'Штрафы, взыскания', 'Проценты кредит ВБ', 'Проценты СИМПЛФИНАНС ООО МКК',
                'Займ ВБ Проценты', 'Кредит сберпроценты', 'Рови факторинг: проценты',
                'Рови факторинг:пени', 'Лизинг, покупка оборудования, помещений', 'Заработная плата'
            ) THEN value ELSE 0 END
        ) AS "Расходы компании"
    FROM expenses
    GROUP BY end_date
),
wfrm_agg AS (
    SELECT
        date_to,
        SUM("Выручка") AS "Выручка",
        SUM("Удержания") AS "Удержания",
        SUM("Логистика") AS "Логистика",
        SUM("Перечисления по кредиту") AS "Перечисления по кредиту",
        SUM("ВП после ВБ") AS "ВП после ВБ"
    FROM weekly_fin_reports_mv
    GROUP BY date_to
)
SELECT
    wfrm.*,
    w."Удержания" / NULLIF(w."Выручка", 0) AS "Удержания_pct",
    w."Логистика" / NULLIF(w."Выручка", 0) AS "Логистика_pct",
    w."Перечисления по кредиту" / NULLIF(w."Выручка", 0) AS "Перечисления по кредиту_pct",
    (w."ВП после ВБ" - e."Расходы компании") AS "Чистая прибыль",
    (w."ВП после ВБ" - e."Расходы компании")
        / NULLIF(w."Выручка", 0) AS "Чистая прибыль_pct",
    e.*
FROM weekly_fin_reports_mv wfrm
LEFT JOIN wfrm_agg w
    ON wfrm.date_to = w.date_to
LEFT JOIN exp e
    ON wfrm.date_to = e.end_date
ORDER BY wfrm.date_to DESC;
    """

In [5]:
df = pd.read_sql(query, engine)

In [7]:
df.columns

Index(['date_to', 'Комиссия ВБ', 'Комиссия ВБ pct', 'К перечислению',
       'Логистика', 'Итого к оплате', 'Выручка', 'Розничная цена со скидкой',
       'Штрафы', 'Хранение', 'Удержания', 'Платная приемка',
       'Перечисления по кредиту', 'К клиенту при отмене',
       'От клиента при отмене', 'От клиента при возврате',
       'К клиенту при продаже', 'Закупочная стоимость продаж',
       'Закупочная стоимость возвратов', 'Закупочная стоимость',
       'Наша доля до вычета себестоимости', 'ВП после ВБ', 'ВП после ВБ pct',
       'Удержания_pct', 'Логистика_pct', 'Перечисления по кредиту_pct',
       'Чистая прибыль', 'Чистая прибыль_pct', 'end_date', 'Расходы офис',
       'Услуги подбора персонала', 'Арендные платежи (офис)',
       'Услуги связи (интернет, телефон)', 'Стоянка', 'Эксплуатация здания',
       'Коммунальные платежи', 'Расчету с поставщиком материалы', 'Карго',
       'Консультационные услуги по оформл', 'Расходы склад',
       'Транспортные расходы, ГСМ', 'Парковка'